# MLOps ДЗ №4 — демонстрация Feast Feature Store

В ноутбуке демонстрируется работа с тремя Feature View:
1. `driver_hourly_stats` — обычная FV (почасовая агрегатная статистика)
2. `driver_trip_stats` — обычная FV (статистика поездок)
3. `driver_efficiency_odfv` — **on-demand** FV (производные признаки на лету)

Перед запуском ноутбука выполните в терминале (см. README):
```bash
python scripts/generate_data.py
cd feature_repo && feast apply
```

In [1]:
import os
from datetime import datetime, timedelta
from pathlib import Path

import pandas as pd
from feast import FeatureStore

# Путь к feature_repo (на 1 уровень выше папки notebooks/)
REPO_PATH = (Path.cwd().parent / "feature_repo").resolve()
print("Feature repo:", REPO_PATH)

store = FeatureStore(repo_path=str(REPO_PATH))
store

Feature repo: /Users/xcore/Documents/УЧЁБА/MLOps_Otus/GitHub_Space/OTUS_MLOps_HomeWorks/hw_04/feature_repo


## 1. Список объектов в Feature Store
Проверим, что `feast apply` успешно зарегистрировал наши Feature Views.

In [2]:
print("Entities:")
for e in store.list_entities():
    print("   -", e.name, "  join_keys=", e.join_keys)

print("\nFeature Views:")
for fv in store.list_feature_views():
    print("   -", fv.name, "  fields=", [f.name for f in fv.schema])

print("\nOn-Demand Feature Views:")
for odfv in store.list_on_demand_feature_views():
    print("   -", odfv.name, "  fields=", [f.name for f in odfv.schema])

Entities:


AttributeError: 'Entity' object has no attribute 'join_keys'

## 2. Offline retrieval — исторические признаки для обучения модели
Готовим entity dataframe (нужно для каждого `driver_id` указать момент времени, на который хотим срез признаков).

In [ ]:
now = datetime.utcnow()

entity_df = pd.DataFrame({
    "driver_id": [1001, 1002, 1003, 1004, 1005],
    "event_timestamp": [
        now - timedelta(hours=2),
        now - timedelta(hours=5),
        now - timedelta(hours=10),
        now - timedelta(hours=24),
        now - timedelta(hours=48),
    ],
})
entity_df

In [ ]:
FEATURES = [
    # обычные FV
    "driver_hourly_stats:conv_rate",
    "driver_hourly_stats:acc_rate",
    "driver_hourly_stats:avg_daily_trips",
    "driver_trip_stats:total_trips",
    "driver_trip_stats:total_distance_km",
    "driver_trip_stats:total_revenue",
    "driver_trip_stats:avg_rating",
    # on-demand FV — считается на лету из всех вышеперечисленных
    "driver_efficiency_odfv:revenue_per_trip",
    "driver_efficiency_odfv:distance_per_trip",
    "driver_efficiency_odfv:effective_conv_score",
    "driver_efficiency_odfv:is_top_driver",
]

training_df = store.get_historical_features(
    entity_df=entity_df,
    features=FEATURES,
).to_df()

training_df

Это и есть готовый датасет для обучения ML-модели. Все признаки автоматически взяты «как было на момент `event_timestamp`» — без data leakage.

## 3. Materialize — заливаем последние значения признаков в online store
Online store используется для инференса в реальном времени.

In [3]:
from datetime import datetime, timezone

# materialize_incremental подтягивает все новые данные в online store
store.materialize_incremental(end_date=datetime.now(timezone.utc))
print("OK — online store обновлён")

Materializing 2 feature views to 2026-05-04 10:06:56+03:00 into the sqlite online store.

driver_trip_stats from 2026-04-27 10:06:56+03:00 to 2026-05-04 10:06:56+03:00:


100%|█████████████████████████████████████████████████████████████| 10/10 [00:00<00:00, 2169.84it/s]


driver_hourly_stats from 2026-05-02 10:06:56+03:00 to 2026-05-04 10:06:56+03:00:


100%|█████████████████████████████████████████████████████████████| 10/10 [00:00<00:00, 5668.74it/s]

OK — online store обновлён


## 4. Online retrieval — низколатентный инференс
Запрашиваем признаки для конкретных водителей «сейчас» — это то, что использует production-сервис при выдаче скоринга.

In [4]:
online_response = store.get_online_features(
    features=FEATURES,
    entity_rows=[
        {"driver_id": 1001},
        {"driver_id": 1002},
        {"driver_id": 1003},
    ],
).to_dict()

pd.DataFrame(online_response)

NameError: name 'FEATURES' is not defined

## 5. Демонстрация on-demand трансформации отдельно
Покажем, что **on-demand FV** действительно вычисляется в момент запроса. Видны новые колонки `revenue_per_trip`, `distance_per_trip`, `effective_conv_score`, `is_top_driver`, которых нет ни в одном из исходных parquet.

In [5]:
odfv_only = store.get_online_features(
    features=[
        "driver_efficiency_odfv:revenue_per_trip",
        "driver_efficiency_odfv:distance_per_trip",
        "driver_efficiency_odfv:effective_conv_score",
        "driver_efficiency_odfv:is_top_driver",
    ],
    entity_rows=[{"driver_id": d} for d in range(1001, 1011)],
).to_df()

odfv_only.sort_values("effective_conv_score", ascending=False)

,driver_id,revenue_per_trip,distance_per_trip,effective_conv_score,is_top_driver
2,1003,365.240469,54.711966,0.724197,1
0,1001,117.652194,4.518751,0.561051,1
9,1010,6.810083,8.211254,0.465689,0
5,1006,26.885032,6.496884,0.371078,0
3,1004,51.089930,3.070607,0.340861,0
1,1002,69.085458,2.828897,0.322639,0
8,1009,65.854086,12.278980,0.319936,0
7,1008,79.808375,3.610943,0.310013,0
6,1007,227.528415,17.198305,0.278539,1
4,1005,281.414059,2.261016,0.275538,0


## Итог
Все три Feature View работают:
- ✅ `driver_hourly_stats`         — historical + online
- ✅ `driver_trip_stats`           — historical + online
- ✅ `driver_efficiency_odfv`     — производные признаки on-the-fly

Задание №4 успешно выполнено.

In [ ]:
X